# LDLR DMS — Roth et al. *Science* 2025

Builds `data/other_benchmarks/ldlr_science_2025.parquet` from the paper's supplementary
data files (`science.ady7186_data_s1` , `_s2`).

| Supp file | assay (as stored in the parquet) |
|---|---|
| Data S1 | `LDL_uptake_functional` |
| Data S2 | `LDLR_cell_surface_abundance` |

Each supp file has the same 7 columns (`hgvs_pro, hgvsp, aapos, score, sd, se, df`).
This notebook concatenates them (S1, S2 order), tags each with its `assay` and the
LDLR Ensembl id, and parses `hgvsp` into a 1-letter `mutant` plus `ref_aa / aa_position /
alt_aa`. Synonymous rows (`hgvsp` like `Gly2=`) and indels (`Gly5del`) don't parse:
`mutant` keeps the raw `hgvsp` string and the three parsed columns stay null (matches the
shipped file: exactly 2 055 such rows).

The final cell asserts the result is byte-identical to the file already on HuggingFace.

In [ ]:
import re
import pathlib
import polars as pl

REPO_ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
                 if (p / 'utils' / 'variant_filtering.py').exists())

# Point this at the unzipped supplementary bundle (science.ady7186_data_s1_to_s8/).
RAW_DIR = pathlib.Path('~/Downloads/science.ady7186_data_s1_to_s8').expanduser()

OUT = REPO_ROOT / 'data' / 'other_benchmarks' / 'ldlr_science_2025.parquet'

In [ ]:
LDLR_REGION = 'ENSG00000130164'

# Supp file -> assay name, in the order the shipped parquet stores them.
FILE_ASSAY = {
    'science.ady7186_data_s1.csv': 'LDL_uptake_functional',
    'science.ady7186_data_s2.csv': 'LDLR_cell_surface_abundance',
}

RAW_COLS = ['hgvs_pro', 'hgvsp', 'aapos', 'score', 'sd', 'se', 'df']
OUT_COLS = RAW_COLS + ['assay', 'region', 'mutant', 'ref_aa', 'aa_position', 'alt_aa']

## Parse `hgvsp`

`hgvsp` is 3-letter protein notation without the `p.` prefix: `Gly2Leu`, `Gly2*`
(nonsense; the raw `hgvs_pro` writes this as `Gly2Ter`), `Gly2=` (synonymous).

* match `^(Xaa)(pos)(Xaa|\*)$` &rarr; 1-letter `ref`, int `pos`, 1-letter `alt`
  (`Ter`/`*` &rarr; `*`), and `mutant = ref + pos + alt`
* anything else (synonymous `=`, indels) &rarr; `mutant = hgvsp` verbatim, parsed
  columns null. `aapos` (the raw column) is kept regardless.

In [ ]:
AA3TO1 = {
    'Ala': 'A', 'Arg': 'R', 'Asn': 'N', 'Asp': 'D', 'Cys': 'C', 'Gln': 'Q',
    'Glu': 'E', 'Gly': 'G', 'His': 'H', 'Ile': 'I', 'Leu': 'L', 'Lys': 'K',
    'Met': 'M', 'Phe': 'F', 'Pro': 'P', 'Ser': 'S', 'Thr': 'T', 'Trp': 'W',
    'Tyr': 'Y', 'Val': 'V', 'Ter': '*',
}
RX = r'^([A-Za-z]{3})([0-9]+)([A-Za-z]{3}|\*)$'


def parse_hgvsp(df: pl.DataFrame) -> pl.DataFrame:
    return (
        df.with_columns(
            _r=pl.col('hgvsp').str.extract(RX, 1),
            _p=pl.col('hgvsp').str.extract(RX, 2),
            _a=pl.col('hgvsp').str.extract(RX, 3),
        )
        .with_columns(
            # map 3-letter -> 1-letter; unmapped tokens (indels like `Gly5del`,
            # regex misses) become null. replace_strict null handling is
            # polars-version-dependent, so guard is_null first.
            _R=pl.when(pl.col('_r').is_null()).then(None)
                 .otherwise(pl.col('_r').replace_strict(AA3TO1, default=None)),
            _A=pl.when(pl.col('_a').is_null()).then(None)
                 .when(pl.col('_a') == '*').then(pl.lit('*'))
                 .otherwise(pl.col('_a').replace_strict(AA3TO1, default=None)),
        )
        .with_columns(
            # a row parses only if BOTH ends are real amino acids (Ter/* counts).
            # substitutions and nonsense keep the parsed columns; synonymous (`=`)
            # and indels (`del`, `ins`, ...) leave ref_aa / aa_position / alt_aa null.
            _ok=pl.col('_R').is_not_null() & pl.col('_A').is_not_null(),
        )
        .with_columns(
            ref_aa=pl.when(pl.col('_ok')).then(pl.col('_R')),
            alt_aa=pl.when(pl.col('_ok')).then(pl.col('_A')),
            aa_position=pl.when(pl.col('_ok')).then(pl.col('_p').cast(pl.Int64)),
        )
        .with_columns(
            mutant=pl.when(pl.col('_ok'))
                     .then(pl.col('ref_aa') + pl.col('aa_position').cast(pl.Utf8) + pl.col('alt_aa'))
                     .otherwise(pl.col('hgvsp')),
        )
        .drop(['_r', '_p', '_a', '_R', '_A', '_ok'])
    )

In [ ]:
parts = []
for fname, assay in FILE_ASSAY.items():
    part = (
        pl.read_csv(RAW_DIR / fname)
        .select(RAW_COLS)                       # guard column order / drop extras
        .with_columns(assay=pl.lit(assay), region=pl.lit(LDLR_REGION))
    )
    parts.append(part)
    print(f'{fname:32s} {assay:36s} {part.height:6d} rows')

ldlr = parse_hgvsp(pl.concat(parts)).select(OUT_COLS)
print(f'total: {ldlr.height} rows')
ldlr.head()

## Verify against the shipped HuggingFace file

In [ ]:
import pyarrow.parquet as pq

new = ldlr.to_arrow()
old = pq.read_table(OUT)

assert new.schema.names == old.schema.names, (new.schema.names, old.schema.names)
assert new.num_rows == old.num_rows == 34186, (new.num_rows, old.num_rows)
assert ldlr['ref_aa'].null_count() == 2055, ldlr['ref_aa'].null_count()

# same-order check: the shipped file is S1,S2 with raw row order preserved
old_pl = pl.from_arrow(old)
row_exact = ldlr.equals(old_pl.select(OUT_COLS))
print('exact row-order match:', row_exact)

# order-independent fallback
key = ['assay', 'hgvs_pro']
assert ldlr.sort(key).equals(old_pl.select(OUT_COLS).sort(key)), 'content differs after sort'
print('content matches the shipped ldlr_science_2025.parquet')

In [ ]:
# Overwrite the shipped file (only when the assert above passes and you mean to).
# ldlr.write_parquet(OUT)